# Real Full 29k Benchmark: GAP + MILP + Genetic

Ноутбук для датасета `src/data/real/full_29k/dataset_real_spb_full_29k.json`.

По умолчанию включён только GAP (`RUN_GAP=True`), а MILP/Genetic выключены, чтобы избежать слишком долгого запуска на 29k задачах.


In [7]:
from __future__ import annotations

from pathlib import Path
import sys
import importlib
import inspect
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Принудительно вычищаем старые flowopt-модули из kernel-кеша
for mod_name in list(sys.modules):
    if mod_name.startswith("flowopt"):
        sys.modules.pop(mod_name, None)

import flowopt.pipeline as fp
fp = importlib.reload(fp)

DATA_ROOT = fp.DATA_ROOT
run_real_gap_vrp = fp.run_real_gap_vrp
run_real_milp = fp.run_real_milp
run_real_genetic = fp.run_real_genetic

milp_sig = inspect.signature(run_real_milp)
if "time_limit_sec" not in milp_sig.parameters:
    raise RuntimeError(f"Stale run_real_milp loaded: signature={milp_sig}")

pd.set_option("display.max_colwidth", 200)
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("pipeline module:", fp.__file__)
print("run_real_milp signature:", milp_sig)


REPO_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows
DATA_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows/src/data
pipeline module: /Users/igoreshka/Desktop/Optimization-of-flows/src/flowopt/pipeline.py
run_real_milp signature: (*, dataset_path: 'Path | str' = PosixPath('/Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/real_simple/dataset_real_spb_simple.json'), time_limit_sec: 'int' = 60, unassigned_penalty: 'float' = 100000.0, show_progress: 'bool' = False, progress_hook: 'Callable[[str], None] | None' = None) -> 'RunMetrics'


In [8]:
import json

REAL_DIR = DATA_ROOT / "real"
FULL_29K_DIR = REAL_DIR / "full_29k"
DATASET_PATH = FULL_29K_DIR / "dataset_real_spb_full_29k.json"
SUMMARY_PATH = FULL_29K_DIR / "summary_real_spb_full_29k.json"
GRAPH_PNG_PATH = FULL_29K_DIR / "graph_real_spb_full_29k.png"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"DATASET_PATH not found: {DATASET_PATH}")

print("DATASET_PATH:", DATASET_PATH)
print("SUMMARY_PATH:", SUMMARY_PATH)
print("GRAPH_PNG_PATH:", GRAPH_PNG_PATH)
print("dataset exists:", DATASET_PATH.exists())
print("summary exists:", SUMMARY_PATH.exists())
print("graph png exists:", GRAPH_PNG_PATH.exists())

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else {}
summary


DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/full_29k/dataset_real_spb_full_29k.json
SUMMARY_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/full_29k/summary_real_spb_full_29k.json
GRAPH_PNG_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/full_29k/graph_real_spb_full_29k.png
dataset exists: True
summary exists: True
graph png exists: True


{'dataset_name': 'real_data_notebook_pack_march_2026_full_29k_af_1',
 'counts': {'road_nodes': 46730,
  'road_edges': 109901,
  'mno': 9228,
  'object1': 9,
  'depots': 4,
  'agents': 626,
  'tasks': 29518,
  'routes': 0},
 'preprocess': {'density_kg_m3': 76.7,
  'tasks_before_sampling': 29518,
  'tasks_after_sampling': 29518,
  'agent_fraction': 1.0,
  'active_transport_rows_before_fraction': 786,
  'active_transport_rows_before_agent_sampling': 626,
  'active_transport_rows': 626,
  'objects': 9,
  'zone_object_edges': 26,
  'total_task_mass_t': 11731.624722999999,
  'total_object_capacity_t': 5424.657534246576,
  'container_to_vehicle_types': {'A': ['VT_A', 'VT_AB', 'VT_ABD', 'VT_AD'],
   'B': ['VT_AB', 'VT_ABD'],
   'C': ['VT_C', 'VT_CD'],
   'D': ['VT_ABD', 'VT_AD', 'VT_CD']},
  'vehicle_profiles': {'VT_A': {'max_daily_km': 280.0,
    'max_shift_hours': 14.0,
    'avg_speed_kmph': 26.0,
    'capacity_median_t': 3.92},
   'VT_AB': {'max_daily_km': 320.0,
    'max_shift_hours': 14.0

In [12]:
# Выбор алгоритмов из списка (для 29k)
# Допустимые значения: "gap_vrp", "real_milp", "real_genetic"
SELECTED_ALGORITHMS = ["gap_vrp"]

# GAP
GAP_STEP1_METHOD = "lp"
GAP_ITER = 20
MAX_AGENTS = None

# Real MILP
RMILP_TIME_LIMIT_SEC = 60
RMILP_UNASSIGNED_PENALTY = 1e5

# Genetic
GA_POPULATION_SIZE = 40
GA_GENERATIONS = 60
GA_ELITE_SIZE = 4
GA_SEED = 42

# Live progress in notebook output
SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = False

from time import perf_counter


def make_progress_logger(enabled: bool):
    if not enabled:
        return None
    t0 = perf_counter()

    def _log(message: str) -> None:
        dt = perf_counter() - t0
        print(f"[+{dt:7.1f}s] {message}", flush=True)

    return _log


progress_log = make_progress_logger(SHOW_ALGO_PROGRESS)


def run_with_live_status(label, fn, **kwargs):
    if progress_log is not None:
        progress_log(f"{label}: START")
    ts = perf_counter()
    out = fn(**kwargs)
    if progress_log is not None:
        progress_log(f"{label}: DONE in {perf_counter() - ts:.1f}s")
    return out


algo_registry = {
    "gap_vrp": lambda: run_with_live_status(
        "GAP-VRP",
        run_real_gap_vrp,
        dataset_path=DATASET_PATH,
        step1_method=GAP_STEP1_METHOD,
        gap_iter=GAP_ITER,
        max_agents=MAX_AGENTS,
        use_repair=True,
        show_progress=SHOW_SOLVER_DETAILS,
        verbose=False,
        progress_hook=progress_log,
    ),
    "real_milp": lambda: run_with_live_status(
        "REAL-MILP",
        run_real_milp,
        dataset_path=DATASET_PATH,
        time_limit_sec=RMILP_TIME_LIMIT_SEC,
        unassigned_penalty=RMILP_UNASSIGNED_PENALTY,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
    "real_genetic": lambda: run_with_live_status(
        "REAL-GENETIC",
        run_real_genetic,
        dataset_path=DATASET_PATH,
        population_size=GA_POPULATION_SIZE,
        generations=GA_GENERATIONS,
        elite_size=GA_ELITE_SIZE,
        seed=GA_SEED,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
}

selected = list(dict.fromkeys(SELECTED_ALGORITHMS))
unknown = [name for name in selected if name not in algo_registry]
if unknown:
    raise ValueError(f"Unknown algorithms in SELECTED_ALGORITHMS: {unknown}")
if not selected:
    raise RuntimeError("SELECTED_ALGORITHMS is empty.")

if progress_log is not None:
    progress_log(f"Selected algorithms: {selected}")

results = [algo_registry[name]() for name in selected]

benchmark_df = pd.DataFrame([r.as_dict() for r in results])
benchmark_df = benchmark_df.sort_values(
    by=["all_checks_ok", "transport_work_ton_km", "total_km", "runtime_sec"],
    ascending=[False, True, True, True],
    na_position="last",
).reset_index(drop=True)

# Save benchmark artifact to local JSON
from datetime import datetime

LOCAL_OUT_DIR = REPO_ROOT / "notebooks" / "local" / "real_data_full_29k"
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULT_JSON_PATH = LOCAL_OUT_DIR / f"benchmark_{RUN_TAG}.json"

records = benchmark_df.where(pd.notna(benchmark_df), None).to_dict(orient="records")
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "real_full_29k_3algo_benchmark.ipynb",
    "dataset_path": str(DATASET_PATH),
    "config": {
        "selected_algorithms": selected,
        "gap_step1_method": GAP_STEP1_METHOD,
        "gap_iter": GAP_ITER,
        "max_agents": MAX_AGENTS,
        "rmilp_time_limit_sec": RMILP_TIME_LIMIT_SEC,
        "rmilp_unassigned_penalty": RMILP_UNASSIGNED_PENALTY,
        "ga_population_size": GA_POPULATION_SIZE,
        "ga_generations": GA_GENERATIONS,
        "ga_elite_size": GA_ELITE_SIZE,
        "ga_seed": GA_SEED,
        "show_algo_progress": SHOW_ALGO_PROGRESS,
        "show_solver_details": SHOW_SOLVER_DETAILS,
    },
    "results": records,
}
RESULT_JSON_PATH.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", RESULT_JSON_PATH)

main_cols = [
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "total_km",
    "deadhead_km",
    "deadhead_share_pct",
    "total_hours",
    "runtime_sec",
    "solver_error",
]
benchmark_df[[c for c in main_cols if c in benchmark_df.columns]]


[+    0.0s] Selected algorithms: ['gap_vrp']
[+    0.0s] GAP-VRP: START
[+    0.0s] [real_gap_vrp] load dataset: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/full_29k/dataset_real_spb_full_29k.json
[+    1.3s] [real_gap_vrp] dataset loaded (nodes=46730, edges=127100, agents=626, tasks=29518)
[+    1.3s] [real_gap_vrp] build nx graph
[+    1.5s] [real_gap_vrp] solver start
[+    1.5s] [real_gap_vrp] [GAP-VRP] start: GAP-Lagrangean + VRP(NN+2opt) [step1=lp]
[+    1.5s] [real_gap_vrp] [GAP-VRP] step1: task generation
[+    1.5s] [real_gap_vrp] [GAP-step1] LP setup: sources=9221, sinks=9, variables=82989, total_waste_tons=11719.199
[+  133.8s] [real_gap_vrp] [GAP-step1] distance matrix: sources=461/9221, elapsed=132.3s
[+  290.0s] [real_gap_vrp] [GAP-step1] distance matrix: sources=922/9221, elapsed=288.5s
[+  431.7s] [real_gap_vrp] [GAP-step1] distance matrix: sources=1383/9221, elapsed=430.2s
[+  569.6s] [real_gap_vrp] [GAP-step1] distance matrix: sources=1844/9221, elaps

KeyboardInterrupt: 

In [10]:
# Детали проверок и параметров по каждому алгоритму
detail_cols = [
    "algorithm",
    "checks",
    "solver_error",
    "step1_method",
    "gap_iter",
    "used_agents",
    "time_limit_sec",
    "unassigned_penalty",
    "population_size",
    "generations",
    "generations_requested",
    "generation_scale",
    "elite_size",
]
benchmark_df[[c for c in detail_cols if c in benchmark_df.columns]]


,algorithm,checks,solver_error,time_limit_sec,unassigned_penalty
0,real_milp,"{'hard_constraints_ok': False, 'daily_limits_ok': False, 'reachability_ok': True, 'all_tasks_assigned': False, 'mno_coverage_ok': False, 'all_checks_ok': False}",RuntimeError: Real MILP failed (status=1): Time limit reached. (HiGHS Status 13: model_status is Time limit reached; primal_status is None),60,100000.0


In [11]:
from flowopt.reporting import solution_breakdown_tables
from IPython.display import Markdown, display

MAX_AGENTS_TO_SHOW = 30
MAX_TASK_IDS_IN_CELL = 12
MAX_TASK_ROWS_TO_SHOW = 400

for _, row in benchmark_df.iterrows():
    algo = row.get("algorithm", "unknown")
    tables = solution_breakdown_tables(
        row,
        max_agents=MAX_AGENTS_TO_SHOW,
        max_task_ids=MAX_TASK_IDS_IN_CELL,
    )

    display(Markdown(f"### {algo}: решение по агентам"))
    display(tables["summary"])

    if tables["agents"].empty:
        print("No active agents in solution")
        continue

    display(tables["agents"])
    display(Markdown(f"**{algo}: задача -> агент (первые {MAX_TASK_ROWS_TO_SHOW} строк)**"))
    display(tables["tasks"].head(MAX_TASK_ROWS_TO_SHOW))


### real_milp: решение по агентам

,algorithm,active_agents,assigned_tasks,total_mass_tons,total_km,total_hours
0,real_milp,0,0,0.0,None,None


No active agents in solution
